In [66]:
import pandas as pd 
import numpy as np 
import ast 

In [67]:
df1=pd.read_csv('trades.csv')

In [68]:
df1.head()

,Port_IDs,Trade_History
0,3925368433214965504,"[{'time': 1718899656000, 'symbol': 'SOLUSDT', ..."
1,4002413037164645377,"[{'time': 1718980078000, 'symbol': 'NEARUSDT',..."
2,3923766029921022977,"[{'time': 1718677164000, 'symbol': 'ETHUSDT', ..."
3,3994879592543698688,"[{'time': 1718678214000, 'symbol': 'ETHUSDT', ..."
4,3926423286576838657,"[{'time': 1718979615000, 'symbol': 'ETHUSDT', ..."


In [69]:
df1["Trade_History"] = df1["Trade_History"].fillna("[]")

In [70]:

# Convert Trade_History string to list of dictionaries
df1["Trade_History"] = df1["Trade_History"].apply(ast.literal_eval)

In [71]:
# Expand Trade_History column (since it's a list of trades per Port_ID)
df_exploded = df1.explode("Trade_History")

In [72]:
# Normalize the nested JSON trade details into separate columns
df_normalized = pd.json_normalize(df_exploded["Trade_History"])

In [73]:
df_final = pd.concat([df_exploded[["Port_IDs"]].reset_index(drop=True), df_normalized], axis=1)


In [74]:
# Normalize the nested JSON trade details into separate columns
df_normalized = pd.json_normalize(df_exploded["Trade_History"])

In [75]:
# Merge with Port_IDs
df_final = pd.concat([df_exploded[["Port_IDs"]].reset_index(drop=True), df_normalized], axis=1)

In [76]:
df_final.head()

,Port_IDs,time,symbol,side,price,fee,feeAsset,quantity,quantityAsset,realizedProfit,realizedProfitAsset,baseAsset,qty,positionSide,activeBuy
0,3925368433214965504,1.718900e+12,SOLUSDT,BUY,132.53700,-0.994027,USDT,1988.05500,USDT,0.0,USDT,SOL,15.0,LONG,True
1,3925368433214965504,1.718900e+12,DOGEUSDT,BUY,0.12182,-0.279796,USDT,1398.98088,USDT,0.0,USDT,DOGE,11484.0,LONG,False
2,3925368433214965504,1.718900e+12,DOGEUSDT,BUY,0.12182,-0.039494,USDT,197.47022,USDT,0.0,USDT,DOGE,1621.0,LONG,False
3,3925368433214965504,1.718900e+12,DOGEUSDT,BUY,0.12182,-0.008284,USDT,16.56752,USDT,0.0,USDT,DOGE,136.0,LONG,True
4,3925368433214965504,1.718900e+12,DOGEUSDT,BUY,0.12182,-0.046109,USDT,92.21774,USDT,0.0,USDT,DOGE,757.0,LONG,True


In [77]:
df_final.to_csv("cleaned_binance_trade_data.csv", index=False)

In [78]:
df = pd.read_csv("cleaned_binance_trade_data.csv")

C:\Users\Charanjot Kaur\AppData\Local\Temp\ipykernel_16616\1726700713.py:1: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("cleaned_binance_trade_data.csv")


In [79]:
# Convert timestamp to readable date
df["time"] = pd.to_datetime(df["time"], unit='ms')


In [80]:
df.head()

,Port_IDs,time,symbol,side,price,fee,feeAsset,quantity,quantityAsset,realizedProfit,realizedProfitAsset,baseAsset,qty,positionSide,activeBuy
0,3925368433214965504,2024-06-20 16:07:36,SOLUSDT,BUY,132.53700,-0.994027,USDT,1988.05500,USDT,0.0,USDT,SOL,15.0,LONG,True
1,3925368433214965504,2024-06-20 16:06:58,DOGEUSDT,BUY,0.12182,-0.279796,USDT,1398.98088,USDT,0.0,USDT,DOGE,11484.0,LONG,False
2,3925368433214965504,2024-06-20 16:06:58,DOGEUSDT,BUY,0.12182,-0.039494,USDT,197.47022,USDT,0.0,USDT,DOGE,1621.0,LONG,False
3,3925368433214965504,2024-06-20 16:06:56,DOGEUSDT,BUY,0.12182,-0.008284,USDT,16.56752,USDT,0.0,USDT,DOGE,136.0,LONG,True
4,3925368433214965504,2024-06-20 16:06:56,DOGEUSDT,BUY,0.12182,-0.046109,USDT,92.21774,USDT,0.0,USDT,DOGE,757.0,LONG,True


In [81]:
account_metrics = df.groupby("Port_IDs").agg(
    PnL=("realizedProfit", "sum"),
    Total_Investment=("quantity", "sum"),
    Win_Positions=("realizedProfit", lambda x: (x > 0).sum()),
    Total_Positions=("realizedProfit", "count")
).reset_index()

In [82]:
#Roi = Return On Investment 
#ROI=((pnl/Total Investment)*100)
account_metrics["ROI"] = (account_metrics["PnL"] / account_metrics["Total_Investment"]) * 100


In [83]:
#  Win Rate (%) = ((win position / total positions )*100)
account_metrics["Win_Rate"]=(account_metrics["Win_Positions"]/account_metrics["Total_Positions"])*100

In [109]:
account_metrics.drop(columns=["Sharpe_Ratio",'MDD'],inplace=True)

In [112]:
# Compute mean and standard deviation of PnL for each Port_ID
sharpe_df = df.groupby("Port_IDs")["realizedProfit"].agg(["mean", "std"])

# Compute Sharpe Ratio (ignore division by zero cases)
sharpe_df["Sharpe_Ratio"] = sharpe_df["mean"] / (sharpe_df["std"] + 1e-9)  # Add small value to avoid division by zero

# Merge back into account_metrics
account_metrics = account_metrics.merge(sharpe_df[["Sharpe_Ratio"]], left_on="Port_IDs", right_index=True, how="left")


In [114]:
def max_drawdown(profits):
    peak = profits.cummax()  # Rolling maximum profit
    drawdown = (profits - peak) / (peak + 1e-9)  # Normalize by peak (avoid div by zero)
    return drawdown.min()  # Worst drawdown

# Compute MDD for each Port_ID on the PnL history
mdd_df = df.groupby("Port_IDs")["realizedProfit"].apply(max_drawdown).reset_index()
mdd_df.rename(columns={"realizedProfit": "MDD"}, inplace=True)

# Merge MDD into account_metrics
account_metrics = account_metrics.merge(mdd_df, on="Port_IDs", how="left")


In [115]:
account_metrics 

,Port_IDs,PnL,Total_Investment,Win_Positions,Total_Positions,ROI,Win_Rate,Sharpe_Ratio,MDD
0,3672754654734989568,566.597660,1.189369e+05,210,474,0.476385,44.303797,0.185274,-1.642161
1,3733192481840423936,2923.977200,1.164472e+06,553,689,0.251099,80.261248,0.060265,-3.663924
2,3768170840939476993,243.668899,2.775560e+03,6,14,8.779089,42.857143,0.424277,-1.000000
3,3784403294629753856,2521.814305,7.421060e+05,1829,6050,0.339819,30.231405,0.106585,-1.829193
4,3786761687746711808,205.021400,6.174001e+04,37,82,0.332072,45.121951,0.215150,-5.331166
...,...,...,...,...,...,...,...,...,...
145,4039279455324236544,1038.807419,1.016345e+05,181,327,1.022101,55.351682,0.471684,-1.000000
146,4040382575336130560,0.000000,1.955943e+04,0,76,0.000000,0.000000,0.000000,0.000000
147,4040843843196854529,2151.704060,2.183311e+05,19,59,0.985523,32.203390,0.341668,-1.000000
148,4041804592937345281,-776.343000,5.781229e+05,85,368,-0.134287,23.097826,-0.077929,-5.130315


In [117]:
sharpe_df = df.groupby("Port_IDs")["realizedProfit"].agg(["mean", "std"])

In [120]:
sharpe_df

,mean,std,Sharpe_Ratio
Port_IDs,,,
3672754654734989568,1.195354,6.451799,0.185274
3733192481840423936,4.243799,70.418447,0.060265
3768170840939476993,17.404921,41.022499,0.424277
3784403294629753856,0.416829,3.910762,0.106585
3786761687746711808,2.500261,11.620986,0.215150
...,...,...,...
4039279455324236544,3.176781,6.734983,0.471684
4040382575336130560,0.000000,0.000000,NaN
4040843843196854529,36.469560,106.739883,0.341668


In [119]:
sharpe_df["Sharpe_Ratio"] = sharpe_df["mean"] / (sharpe_df["std"])

In [125]:
account_metrics = account_metrics.merge(
    sharpe_df[["Sharpe_Ratio"]], left_on="Port_IDs", right_index=True, how="left"
)


In [128]:
account_metrics.drop(columns=["Sharpe_Ratio_y"],inplace=True)

In [129]:
account_metrics

,Port_IDs,PnL,Total_Investment,Win_Positions,Total_Positions,ROI,Win_Rate,Sharpe_Ratio_x,MDD
0,3672754654734989568,566.597660,1.189369e+05,210,474,0.476385,44.303797,0.185274,-1.642161
1,3733192481840423936,2923.977200,1.164472e+06,553,689,0.251099,80.261248,0.060265,-3.663924
2,3768170840939476993,243.668899,2.775560e+03,6,14,8.779089,42.857143,0.424277,-1.000000
3,3784403294629753856,2521.814305,7.421060e+05,1829,6050,0.339819,30.231405,0.106585,-1.829193
4,3786761687746711808,205.021400,6.174001e+04,37,82,0.332072,45.121951,0.215150,-5.331166
...,...,...,...,...,...,...,...,...,...
145,4039279455324236544,1038.807419,1.016345e+05,181,327,1.022101,55.351682,0.471684,-1.000000
146,4040382575336130560,0.000000,1.955943e+04,0,76,0.000000,0.000000,0.000000,0.000000
147,4040843843196854529,2151.704060,2.183311e+05,19,59,0.985523,32.203390,0.341668,-1.000000
148,4041804592937345281,-776.343000,5.781229e+05,85,368,-0.134287,23.097826,-0.077929,-5.130315


In [ ]:
# ----------------------------------- Ranking Algorithm for Accounts ----------------------------------- #

In [ ]:
''' Step 1: Define Ranking Criteria
To rank the accounts, we need to decide which metrics are most important. A balanced ranking system might include:

1️ Return on Investment (ROI) → Measures profitability. Higher is better.
2️ Sharpe Ratio → Measures risk-adjusted returns. Higher is better.
3️ Win Rate → Percentage of winning trades. Higher is better.
4️ Maximum Drawdown (MDD) → Measures worst portfolio loss. Lower (less negative) is better.
5️ PnL (Profit & Loss) → Measures total profit. Higher is better.      '''

In [ ]:
''' Step 2: Normalize the Metrics
Since each metric has different units and scales, we should normalize them before combining them into a final score.

 Min-Max Normalization -> We use Min-Max Scaling to bring all values between 0 and 1:

Normalized Value = Metric − Min Value / Max Value − Min Value  '''

In [137]:
# Define the metrics to normalize
metrics_to_normalize = ["ROI", "Sharpe_Ratio_x", "Win_Rate", "PnL"]
inverse_metrics = ["MDD"]  # MDD should be minimized, so we take -MDD for proper scaling

# Normalize selected metrics ->  Normalized Value formula = Metric − Min Value / Max Value − Min Value  
for metric in metrics_to_normalize:
    account_metrics[f"{metric}_normalized"] = (account_metrics[metric] - account_metrics[metric].min()) / \
    (account_metrics[metric].max() - account_metrics[metric].min())

for metric in inverse_metrics:
    account_metrics[f"{metric}_normalized"] = (account_metrics[metric].max() - account_metrics[metric]) / \
                                              (account_metrics[metric].max() - account_metrics[metric].min())


In [132]:
account_metrics

,Port_IDs,PnL,Total_Investment,Win_Positions,Total_Positions,ROI,Win_Rate,Sharpe_Ratio_x,MDD,ROI_normalized,Sharpe_Ratio_x_normalized,Win_Rate_normalized,PnL_normalized,MDD_normalized
0,3672754654734989568,566.597660,1.189369e+05,210,474,0.476385,44.303797,0.185274,-1.642161,0.053635,0.349526,0.551995,0.153100,2.797791e-12
1,3733192481840423936,2923.977200,1.164472e+06,553,689,0.251099,80.261248,0.060265,-3.663924,0.035416,0.183518,1.000000,0.181049,6.242320e-12
2,3768170840939476993,243.668899,2.775560e+03,6,14,8.779089,42.857143,0.424277,-1.000000,0.725084,0.666914,0.533971,0.149271,1.703725e-12
3,3784403294629753856,2521.814305,7.421060e+05,1829,6050,0.339819,30.231405,0.106585,-1.829193,0.042591,0.245029,0.376663,0.176281,3.116442e-12
4,3786761687746711808,205.021400,6.174001e+04,37,82,0.332072,45.121951,0.215150,-5.331166,0.041965,0.389200,0.562189,0.148813,9.082844e-12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145,4039279455324236544,1038.807419,1.016345e+05,181,327,1.022101,55.351682,0.471684,-1.000000,0.097768,0.729868,0.689644,0.158698,1.703725e-12
146,4040382575336130560,0.000000,1.955943e+04,0,76,0.000000,0.000000,0.000000,0.000000,0.015110,0.103487,0.000000,0.146382,0.000000e+00
147,4040843843196854529,2151.704060,2.183311e+05,19,59,0.985523,32.203390,0.341668,-1.000000,0.094810,0.557211,0.401232,0.171893,1.703725e-12
148,4041804592937345281,-776.343000,5.781229e+05,85,368,-0.134287,23.097826,-0.077929,-5.130315,0.004250,0.000000,0.287783,0.137178,8.740649e-12


In [ ]:
# ----------------------------- Step 3: Weighted Scoring System ------------------------

In [139]:
# Define weights
weights = {
    "ROI_normalized": 0.30,
    "Sharpe_Ratio_x_normalized": 0.25,
    "Win_Rate_normalized": 0.20,
    "MDD_normalized": 0.15,  # Lower MDD is better
    "PnL_normalized": 0.10
}

# Compute final ranking score using weighted sum
account_metrics["Final_Score"] = sum(account_metrics[metric] * weight for metric, weight in weights.items())

# Rank accounts based on Final Score (Descending Order)
account_metrics = account_metrics.sort_values(by="Final_Score", ascending=False)



In [147]:
account_metrics

,Port_IDs,PnL,Total_Investment,Win_Positions,Total_Positions,ROI,Win_Rate,Sharpe_Ratio_x,MDD,ROI_normalized,Sharpe_Ratio_x_normalized,Win_Rate_normalized,PnL_normalized,MDD_normalized,Final_Score
8,3826087012661391104,532.656974,4.373742e+03,63,108,12.178517,58.333333,0.675101,-1.334096,1.000000,1.000000,0.726793,0.152697,2.272934e-12,0.710628
2,3768170840939476993,243.668899,2.775560e+03,6,14,8.779089,42.857143,0.424277,-1.000000,0.725084,0.666914,0.533971,0.149271,1.703725e-12,0.505975
48,3956048468100538880,1373.564890,1.232382e+05,20,28,1.114561,71.428571,0.524862,-1.000000,0.105246,0.800487,0.889951,0.162667,1.703725e-12,0.425952
144,4039129759104249600,1264.289200,3.997778e+04,59,133,3.162479,44.360902,0.522043,-1.000000,0.270863,0.796743,0.552706,0.161372,1.703725e-12,0.407123
16,3891020560590657281,2856.300564,1.638344e+05,283,437,1.743407,64.759725,0.432477,-1.000000,0.156101,0.677803,0.806862,0.180246,1.703725e-12,0.395678
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9,3858510226868015873,-280.972950,7.873928e+05,243,996,-0.035684,24.397590,-0.013558,-14.521672,0.012224,0.085482,0.303977,0.143051,2.474094e-11,0.100138
148,4041804592937345281,-776.343000,5.781229e+05,85,368,-0.134287,23.097826,-0.077929,-5.130315,0.004250,0.000000,0.287783,0.137178,8.740649e-12,0.072549
72,3995532094997544704,-12346.682219,6.608267e+06,1295,6051,-0.186837,21.401421,-0.064248,-3.219149,0.000000,0.018169,0.266647,0.000000,5.484546e-12,0.057872
146,4040382575336130560,0.000000,1.955943e+04,0,76,0.000000,0.000000,0.000000,0.000000,0.015110,0.103487,0.000000,0.146382,0.000000e+00,0.045043


In [145]:
top_20_accounts = account_metrics[["Port_IDs", "Final_Score"]].head(20).reset_index()
top_20_accounts

,index,Port_IDs,Final_Score
0,8,3826087012661391104,0.710628
1,2,3768170840939476993,0.505975
2,48,3956048468100538880,0.425952
3,144,4039129759104249600,0.407123
4,16,3891020560590657281,0.395678
5,63,3986814617275053313,0.389964
6,101,4022641794255717633,0.386400
7,134,4035430878731345664,0.370646
8,145,4039279455324236544,0.365596
9,37,3943533600390906881,0.352262


In [146]:
top_20_accounts.to_csv("top_20_accounts.csv", index=False)


In [148]:
account_metrics.to_csv("account_metrics.csv", index=False)
